## Kitchener Water Main Breaks + Water Mains
### Cleaning & Initial preprocessing (Joe Tambeana)
The goal is
- to understand the datatypes that each dataset holds
- clean and drop out columns or rows that are not relevant for analysis
- identify where they can be linked for analysis
- get all cleaned ready for preprocessing


In [2]:
import pandas as pd

kitchenerWaterMainBreaksDf = pd.read_csv("../data/raw/Kitchener/Kitchener_watermainbreaks.csv")
kitchenerWaterMainsDf = pd.read_csv("../data/raw/Kitchener/Kitchener_watermains.csv")

# print the first 10 rows of each dataset
print("=== Kitchener Water Main Breaks Dataset ===")
print(kitchenerWaterMainBreaksDf.head(10).to_string())
print("\n")
print("=== Kitchener Water Mains Dataset ===")
print(kitchenerWaterMainsDf.head(10).to_string())

=== Kitchener Water Main Breaks Dataset ===
   OBJECTID  Wat Break Incident ID           Incident date Type of Asset Broken Does the road need to be closed? Does the sidewalk need to be closed? Estimated Hours for Repair Estimated Number of Units Impacted  CW Service Request Number Current status of the break Status last updated date  CW Workorder # Date operations was returned to normal service              Nature of Break Apparent cause of break Repair Type  Type of Planned Maintenance                                  List Valves Closed  List Valves Opened List Hydrants Called Out List Hydrants Called Back In Categorization of the Break  Road Segment ID Closest Civic Number           Street  Related Asset ID  Related Asset Depth (m)  Depth of Frost (m)  Asset Size (cm)  Year Asset Installed Asset Material Asset Exists                              GLOBALID UPDATE_BY            UPDATE_DATE            x             y
0         1                   2252    12/1/2017 3:15:00 PM            

### Overview of the 2 datasets

In [3]:
print("\nKitchener Water Main Breaks info:")
kitchenerWaterMainBreaksDf.info()
print("\nKitchener Water Mains info:")
kitchenerWaterMainsDf.info()


Kitchener Water Main Breaks info:
<class 'pandas.DataFrame'>
RangeIndex: 2994 entries, 0 to 2993
Data columns (total 37 columns):
 #   Column                                          Non-Null Count  Dtype  
---  ------                                          --------------  -----  
 0   OBJECTID                                        2994 non-null   int64  
 1   Wat Break Incident ID                           2994 non-null   int64  
 2   Incident date                                   2994 non-null   str    
 3   Type of Asset Broken                            2994 non-null   str    
 4   Does the road need to be closed?                540 non-null    str    
 5   Does the sidewalk need to be closed?            537 non-null    str    
 6   Estimated Hours for Repair                      538 non-null    str    
 7   Estimated Number of Units Impacted              525 non-null    str    
 8   CW Service Request Number                       25 non-null     float64
 9   Current status of

### Check statistics for missing values

In [4]:
# print the count and percentage of missing values for each column

# Define constants
MISSING_COUNT = 'Missing Count'
MISSING_PERCENT = 'Missing %'
NON_NULL_COUNT = 'Non-Null Count'

# using the dataframe already loaded for Kitchener datasets
print("\n=== Kitchener Missing Values Analysis ===")
dfKWaterMainBreaks = kitchenerWaterMainBreaksDf
missing = pd.DataFrame({
    MISSING_COUNT: dfKWaterMainBreaks.isna().sum(),
    MISSING_PERCENT: (dfKWaterMainBreaks.isna().sum() / len(dfKWaterMainBreaks) * 100).round(2),
    NON_NULL_COUNT: dfKWaterMainBreaks.notna().sum()
})
print(missing)

# using the dataframe already loaded for Kitchener datasets
print("\n=== Kitchener Water Mains Missing Values Analysis ===")

dfKWatermains = kitchenerWaterMainsDf
missing = pd.DataFrame({
    MISSING_COUNT: dfKWatermains.isna().sum(),
    MISSING_PERCENT: (dfKWatermains.isna().sum() / len(dfKWatermains) * 100).round(2),
    NON_NULL_COUNT: dfKWatermains.notna().sum()
})
print(missing)



=== Kitchener Missing Values Analysis ===
                                                Missing Count  Missing %  \
OBJECTID                                                    0       0.00   
Wat Break Incident ID                                       0       0.00   
Incident date                                               0       0.00   
Type of Asset Broken                                        0       0.00   
Does the road need to be closed?                         2454      81.96   
Does the sidewalk need to be closed?                     2457      82.06   
Estimated Hours for Repair                               2456      82.03   
Estimated Number of Units Impacted                       2469      82.46   
CW Service Request Number                                2969      99.16   
Current status of the break                                 0       0.00   
Status last updated date                                  151       5.04   
CW Workorder #                               

### Check for negative values

In [5]:
# Check for negative values in numeric columns of Kitchener Water Main Breaks dataset
kMainBreaksnegatives = (dfKWaterMainBreaks.select_dtypes(include='number') < 0).sum().sum()
kMainBreaksnegatives

# Check for negative values in numeric columns of Kitchener Water Mains dataset
kWaterMainBreaks = (dfKWatermains.select_dtypes(include='number') < 0).sum().sum()
kWaterMainBreaks

print(f"Total negative values in Kitchener Water Main Breaks dataset: {kMainBreaksnegatives}")
print(f"Total negative values in Kitchener Water Mains dataset: {kWaterMainBreaks}")

Total negative values in Kitchener Water Main Breaks dataset: 1
Total negative values in Kitchener Water Mains dataset: 3297


### Duplicates check

In [6]:
print("\n=== Kitchener Water Main Breaks Duplicate Check ===")

# Total number of duplicate rows
total_duplicates = dfKWaterMainBreaks.duplicated().sum()
print(f"Total duplicate rows: {total_duplicates}")  

# Percentage of duplicates
if len(dfKWaterMainBreaks) > 0:
    duplicate_pct = (total_duplicates / len(dfKWaterMainBreaks) * 100).round(2)
    print(f"Duplicate percentage: {duplicate_pct}%")

print("\n=== Kitchener Water Mains Duplicate Check ===")

# Total number of duplicate rows
total_duplicates = dfKWatermains.duplicated().sum()
print(f"Total duplicate rows: {total_duplicates}")

# Percentage of duplicates
if len(dfKWatermains) > 0:
    duplicate_pct = (total_duplicates / len(dfKWatermains) * 100).round(2)
    print(f"Duplicate percentage: {duplicate_pct}%")


=== Kitchener Water Main Breaks Duplicate Check ===
Total duplicate rows: 0
Duplicate percentage: 0.0%

=== Kitchener Water Mains Duplicate Check ===
Total duplicate rows: 0
Duplicate percentage: 0.0%


### Data Cleaning process & Initial preprocessing

In [7]:
# # # ============================== Kitchener Water Main Breaks Cleaning =====================================
def clean_data(df):
    """
    Cleans the Kitchener water main breaks dataset by:
    - Dropping irrelevant or mostly empty columns
    - Dropping rows with missing values in key columns
    - Converting data types and creating consistent column names
    """

    # ========================== Drop Irrelevant Columns ==========================
    columns_to_drop = [
        'Does the road need to be closed?',
        'Does the sidewalk need to be closed?',
        'Estimated Hours for Repair',
        'Estimated Number of Units Impacted',
        'CW Service Request Number',
        'Date operations was returned to normal service',
        'Repair Type',
        'Type of Planned Maintenance',
        'List Valves Closed',
        'List Valves Opened',
        'List Hydrants Called Out',
        'List Hydrants Called Back In',
        'Related Asset Depth (m)',
        'Depth of Frost (m)',
        'GLOBALID',
        'Street',                    
        'Asset Exists',
        'OBJECTID',
        'UPDATE_BY',
        'UPDATE_DATE',
        'Wat Break Incident ID',
        'CW Workorder #',
        'Current status of the break',
        'Categorization of the Break',
        'Road Segment ID',
        'Closest Civic Number',
        'Status last updated date'
    ]

    # Drop safely
    df = df.drop(columns=columns_to_drop, errors='ignore')

    # ========================== Drop Rows with Missing Critical Data ==========================
    critical_columns = [
        'Asset Material',
        'Year Asset Installed',
        'Asset Size (cm)',
        'Apparent cause of break',
        'Nature of Break'
    ]

    # Only consider critical columns that exist in the dataset
    existing_critical = [c for c in critical_columns if c in df.columns]
    df = df.dropna(subset=existing_critical)

    # ========================== Data Type Conversions ==========================
    
    # Convert 'Year Asset Installed' to numeric and create 'Installed_Year'
    if 'Year Asset Installed' in df.columns:
        df['Installed_Year'] = (
            pd.to_numeric(df['Year Asset Installed'], errors='coerce')
            .astype('Int64')
        )
        df = df.drop(columns=['Year Asset Installed'])

    # ========================== Rename Columns ==========================
    rename_map = {
        'Asset Material': 'Material',
        'Related Asset ID': 'Pipe_ID',
        'Apparent cause of break': 'Cause_of_Break',
        'Nature of Break': 'Nature_of_Break',
        'Type of Asset Broken': 'Asset_Type',
        'Asset Size (cm)': 'Pipe_Size_cm',
        'x': 'X',
        'y': 'Y'
    }

    # Only rename columns that exist in the dataset
    rename_map = {k: v for k, v in rename_map.items() if k in df.columns}
    df = df.rename(columns=rename_map)

    # ========================== Handling dateline conversion ==========================

    # Convert 'Incident date' to datetime and create 'Break_Date'
    if 'Incident date' in df.columns:
        df = df.rename(columns={'Incident date': 'Break_Date'})
    
    # Convert 'Break_Date' to datetime with specified format, coerce errors to NaT
    df['Break_Date'] = pd.to_datetime(
        df['Break_Date'], 
        format='%m/%d/%Y %I:%M:%S %p', 
        errors='coerce'
    )
    
    # Keep only the date (strip time)
    df['Break_Date'] = df['Break_Date'].dt.date
    df['Break_Date'] = pd.to_datetime(df['Break_Date'])
    
    # =====================Requested Features Creation =====================
    # Create Break_Year
    df['Break_Year'] = df['Break_Date'].dt.year.astype('Int64')

    # Age at Break
    df['Break_Age'] = df['Break_Year'] - df['Installed_Year']

    # Age Squared (for non-linear effects)
    df['Age_Squared'] = df['Break_Age'] ** 2

    # Standardize Leak Type
    cause_mapping = {
        'Full Circle Break': 'Circumferential',
        'Full Circle': 'Circumferential',
        'Longitudinal Break': 'Longitudinal',
        'Leaking Joint': 'Joint',
        'Blow-out': 'Blowout',
        'Corrosion': 'Corrosion',
        'Tap Failure': 'Tap',
        'Crack': 'Crack',
        'Leak Outside Old Repair Band': 'Other',
        'Leak on Unused Fire Line': 'Other',
        'Crack - Main hit by Contractor': 'Crack',
    }
    df['Cause_of_Break_Clean'] = df['Cause_of_Break'].replace(cause_mapping)

    # Remove rows with negative ages (if any)
    df = df[df["Break_Age"] >= 0].copy()
    
    return df

#Apply the cleaning function
df_clean_kitchenerWaterMainBreaks = clean_data(dfKWaterMainBreaks.copy())

df_clean_kitchenerWaterMainBreaks.to_csv(
    "../data/processed/Kitchener_watermain_breaks_cleaned_final.csv", 
    index=False
)

print("Cleaned Kitchener Water Main Breaks Dataset (first 10 rows):")
print(df_clean_kitchenerWaterMainBreaks.head(10).to_string(index=False))

Cleaned Kitchener Water Main Breaks Dataset (first 10 rows):
Break_Date Asset_Type             Nature_of_Break Cause_of_Break  Pipe_ID  Pipe_Size_cm Material           X            Y  Installed_Year  Break_Year  Break_Age  Age_Squared Cause_of_Break_Clean
2017-12-01       MAIN CORROSION AND FITTING/JOINT            AGE   134292         450.0       CI 541741.1448 4812353.8463            1937        2017         80         6400                  AGE
2001-03-26    SERVICE                     UNKNOWN          OTHER  4101323          13.0      XXX 539253.7576 4807874.5871            1965        2001         36         1296                OTHER
2006-09-06    SERVICE                     UNKNOWN          OTHER  4099987          25.0      XXX 545329.4589 4810392.1000            1967        2006         39         1521                OTHER
2006-09-11    SERVICE                     UNKNOWN          OTHER  4642530          25.0      PVC 539592.5208 4808291.4459            1964        2006         4

### Statistics - Kitchener water main breaks after cleaning

In [8]:
# ====================== Kitchener water main breaks quick validation ======================
print("=== Kitchener Feature Creation Summary ===")
print(df_clean_kitchenerWaterMainBreaks[['Break_Date', 'Installed_Year', 'Break_Year', 
                        'Break_Age', 'Age_Squared', 
                        'Cause_of_Break','Cause_of_Break_Clean']].head(10).to_string())

print("\nAge_at_Break Statistics:")
print(df_clean_kitchenerWaterMainBreaks['Break_Age'].describe())

print("\nCause_of_Break Distribution:")
print(df_clean_kitchenerWaterMainBreaks['Cause_of_Break'].value_counts())

# Check for negative ages - should be very few or zero
print(f"\nNumber of negative ages: {(df_clean_kitchenerWaterMainBreaks['Break_Age'] < 0).sum()}")

=== Kitchener Feature Creation Summary ===
   Break_Date  Installed_Year  Break_Year  Break_Age  Age_Squared Cause_of_Break Cause_of_Break_Clean
0  2017-12-01            1937        2017         80         6400            AGE                  AGE
1  2001-03-26            1965        2001         36         1296          OTHER                OTHER
2  2006-09-06            1967        2006         39         1521          OTHER                OTHER
3  2006-09-11            1964        2006         42         1764          OTHER                OTHER
4  2000-01-27            1967        2000         33         1089          OTHER                OTHER
5  1998-08-11            1974        1998         24          576          OTHER                OTHER
6  2006-05-23            1976        2006         30          900          OTHER                OTHER
9  2006-06-26            1975        2006         31          961          OTHER                OTHER
11 2005-06-23            1976        20

In [9]:
# ============================== Kitchener Water Mains Cleaning =====================================
def clean_data(df):
    # Drop columns: 'LINED_DATE', 'CONSULTANT', 'BRIDGE_DETAILS'
    df = df.drop(columns=['OBJECTID', 
                          'MAP_LABEL', 
                          'ROADSEGMENTID', 
                          'ACQUISITION', 
                          'OWNERSHIP',
                          'BRIDGE_MAIN',
                          'REL_CLEANING_AREA',
                          'CLEANED','CATEGORY'])
    
    # rename columns for consistency
    df = df.rename(columns={
        'WATMAINID': 'Pipe_ID',
        'PIPE_SIZE': 'Pipe_Size_mm',
        'MATERIAL': 'Material',
        'INSTALLATION_DATE': 'Install_Date',
        'PRESSURE_ZONE': 'Pressure_Zone',
        'Condition Score': 'Condition_Score',
        'Shape__Length': 'Length_m',
        'Shallow Main': 'Is_Shallow',
    })

    # Drop original columns that have missing values >70% or are low relevance
    df = df.drop(columns=['LINED_DATE', 'CONSULTANT', 'BRIDGE_DETAILS'])

    # ========================== Data Type Conversions ==========================
     
    if 'Install_Date' in df.columns:
        df = df.rename(columns={'Install_Date': 'Install_Date'})
    
    # Convert 'Install_Date' to datetime with specified format, coerce errors to NaT
    df['Install_Date'] = pd.to_datetime(
        df['Install_Date'], 
        format='%m/%d/%Y %I:%M:%S %p', 
        errors='coerce'
    )
    
    # Keep only the date (strip time)
    df['Install_Date'] = df['Install_Date'].dt.date
    df['Install_Date'] = pd.to_datetime(df['Install_Date'])

    # =====================Requested Features Creation =====================
    # Create Install_Year
    df['Install_Year'] = df['Install_Date'].dt.year.astype('Int64')

    df['Condition_Score'] = df['Condition_Score'].replace(-1.0, pd.NA)
    
    df[df.isna().all(axis=1)]
    print(f"Rows with all values missing: {len(df[df.isna().all(axis=1)])}\n")
    return df

df_clean_kitchenerWaterMains = clean_data(dfKWatermains.copy())

df_clean_kitchenerWaterMains.to_csv(
    "/Users/homedesk/Documents/S779/2026/SIT764/capstone/Datasets/Kitchener_watermains_cleaned_final.csv", 
    index=False
)

df_clean_kitchenerWaterMains.head(10)



Rows with all values missing: 0



,Pipe_ID,STATUS,Pressure_Zone,Pipe_Size_mm,Material,LINED,LINED_MATERIAL,Install_Date,CRITICALITY,REL_CLEANING_SUBAREA,Undersized,Is_Shallow,Condition_Score,OVERSIZED,Length_m,Install_Year
0,10080,ACTIVE,KIT 6,450,DI,NO,NONE,1979-01-01,6,8,N,N,8.5,N,34.210563,1979
1,76299,ACTIVE,KIT 4,300,DI,NO,NONE,1968-07-01,7,4,N,N,8.5,N,0.355118,1968
2,10110,ACTIVE,KIT 6,450,DI,NO,NONE,1979-01-01,7,8,N,N,6.1,N,67.852910,1979
3,82566,ACTIVE,KIT 6,450,DI,NO,NONE,1979-01-01,6,8,N,N,4.58,N,7.039328,1979
4,82568,ACTIVE,KIT 6,450,DI,NO,NONE,1979-01-01,6,8,N,N,8.5,N,14.956131,1979
5,10090,ACTIVE,KIT 6,450,DI,NO,NONE,1979-01-01,6,8,N,N,8.5,N,38.526891,1979
6,82570,ACTIVE,KIT 6,450,DI,NO,NONE,1979-01-01,6,7,N,N,8.5,N,12.451964,1979
7,10030,ACTIVE,KIT 6,450,DI,NO,NONE,1979-01-01,7,7,N,N,8.5,N,124.970065,1979
8,10100,ACTIVE,KIT 6,450,DI,NO,NONE,1979-01-01,6,8,N,N,8.5,N,55.671581,1979
9,81620,ACTIVE,KIT 2W,300,PVC,NO,NONE,2015-01-06,0,46,N,N,9.35,N,63.948585,2015


In [10]:
print("\n=== Cleaned Kitchener Water Main Breaks Dataset ===")
print(df_clean_kitchenerWaterMainBreaks.head(10).to_string())

print("\n=== Cleaned Kitchener Water Mains Dataset ===")
print(df_clean_kitchenerWaterMains.head(10).to_string())


=== Cleaned Kitchener Water Main Breaks Dataset ===
   Break_Date Asset_Type              Nature_of_Break Cause_of_Break  Pipe_ID  Pipe_Size_cm Material            X             Y  Installed_Year  Break_Year  Break_Age  Age_Squared Cause_of_Break_Clean
0  2017-12-01       MAIN  CORROSION AND FITTING/JOINT            AGE   134292         450.0       CI  541741.1448  4.812354e+06            1937        2017         80         6400                  AGE
1  2001-03-26    SERVICE                      UNKNOWN          OTHER  4101323          13.0      XXX  539253.7576  4.807875e+06            1965        2001         36         1296                OTHER
2  2006-09-06    SERVICE                      UNKNOWN          OTHER  4099987          25.0      XXX  545329.4589  4.810392e+06            1967        2006         39         1521                OTHER
3  2006-09-11    SERVICE                      UNKNOWN          OTHER  4642530          25.0      PVC  539592.5208  4.808291e+06            1964